In [ ]:
# ===============================
# Feature Analysis (Diabetes Dataset)
# ===============================

# --- Google Drive Mount (สำหรับ Colab) ---
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
data_visualization_path = "/content/drive/MyDrive/Colab Notebooks/selected-topics/data-visualization"

# Load dataset
df = pd.read_csv(f"{data_visualization_path}/diabetes.csv")


# ===============================
# 1️⃣ Quick Model Run (Logistic Regression)
# ===============================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# แบ่ง Features / Labels
X = df.drop("Outcome", axis=1)   # Features
y = df["Outcome"]                # Target / Label

# Train/Test Split (30% test, random_state = 42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# สร้างและเทรน Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate Accuracy
accuracy_score_before_fa = accuracy_score(y_test, y_pred)
print("Accuracy before feature analysis (all features):", accuracy_score_before_fa)


# ===============================
# 2️⃣ Feature Analysis
# ===============================

# ดูสถิติพื้นฐานของ DataFrame
df.describe()

# ตรวจสอบ missing values
df.isnull().sum()

# Correlation Heatmap (ดูความสัมพันธ์ระหว่าง features)
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm")
plt.show()

# Outlier Detection: Boxplot
sns.boxplot(data=df)
plt.xticks(rotation=90)
plt.show()


# ===============================
# 3️⃣ Feature Engineering
# ===============================

# แทนค่าที่เป็น 0 (ไม่สมเหตุสมผล) ด้วยค่า mean ของ column
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in cols:
    df[col] = df[col].replace(0, df[col].mean())

# Scaling features (Standardization)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.drop("Outcome", axis=1))

# แปลงกลับเป็น DataFrame เพื่ออ่านง่าย
df_scaled = pd.DataFrame(X_scaled, columns=df.drop("Outcome", axis=1).columns)


# ===============================
# 4️⃣ Model Run หลัง Scaling
# ===============================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, df["Outcome"], test_size=0.3, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy_score_with_scaling = accuracy_score(y_test, y_pred)
print("Accuracy with scaling:", accuracy_score_with_scaling)


# ===============================
# 5️⃣ Feature Selection (SelectKBest)
# ===============================

from sklearn.feature_selection import SelectKBest, f_classif

# เลือก 5 ฟีเจอร์ที่สัมพันธ์กับ Outcome สูงสุด
selector = SelectKBest(score_func=f_classif, k=5)
X_new = selector.fit_transform(X_scaled, y)

# ดูชื่อฟีเจอร์ที่ถูกเลือก
selected_mask = selector.get_support()
selected_features = df.drop("Outcome", axis=1).columns[selected_mask]
print("Selected features:", list(selected_features))

# แสดง F-score และ p-value ของทุกฟีเจอร์
selector_all = SelectKBest(score_func=f_classif, k='all')
selector_all.fit(X_scaled, y)
feature_scores = pd.DataFrame({
    'Feature': df.drop("Outcome", axis=1).columns,
    'F-score': selector_all.scores_,
    'p-value': selector_all.pvalues_
}).sort_values(by='F-score', ascending=False)
print(feature_scores)


# ===============================
# 6️⃣ Model Run หลัง Feature Selection
# ===============================

X_train, X_test, y_train, y_test = train_test_split(
    X_new, y, test_size=0.3, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(y_test)

accuracy_score_with_selected_features = accuracy_score(y_test, y_pred)
print(f"Accuracy with selected features: {accuracy_score_with_selected_features}")


# ===============================
# 7️⃣ Visualization
# ===============================

# Pairplot: ดูความสัมพันธ์ทีละคู่
sns.pairplot(df, hue="Outcome")

# Scatterplot (ตัวอย่างสำคัญ: Glucose vs Age)
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=df,
    x="Glucose",
    y="Age",
    hue="Outcome",
    alpha=0.7
)
plt.title("Glucose vs Age (Colored by Outcome)")
plt.xlabel("Glucose")
plt.ylabel("Age")
plt.tight_layout()
plt.show()


# ===============================
# 8️⃣ Decision Boundary Plot (Logistic Regression)
# ===============================

import numpy as np
from matplotlib.lines import Line2D

# ใช้ 2 features: Glucose และ Age
X_plot = df[['Glucose','Age']].values
y_plot = df['Outcome'].values

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_plot)

# Train Logistic Regression
model = LogisticRegression()
model.fit(X_scaled, y_plot)

# สร้าง Meshgrid สำหรับ boundary
glucose_min, glucose_max = X_scaled[:,0].min()-1, X_scaled[:,0].max()+1
age_min, age_max = X_scaled[:,1].min()-1, X_scaled[:,1].max()+1
xx, yy = np.meshgrid(
    np.linspace(glucose_min, glucose_max, 200),
    np.linspace(age_min, age_max, 200)
)

# Predict decision boundary
Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Plot
plt.figure(figsize=(8,6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')

sns.scatterplot(
    x=X_scaled[:,0], y=X_scaled[:,1],
    hue=y_plot,
    palette={0: "tab:blue", 1: "tab:orange"},
    alpha=0.8,
    legend=False
)

plt.title("Decision Boundary: Glucose vs Age (Logistic Regression)")
plt.xlabel("Scaled Glucose")
plt.ylabel("Scaled Age")

# Custom legend
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='tab:blue', markersize=8, label='0 = No Diabetes'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='tab:orange', markersize=8, label='1 = Diabetes')
]
plt.legend(handles=legend_elements, title="Outcome")
plt.show()